In [50]:
import json
import joblib
import pandas as pd
import numpy as np
from typing import Any
from pathlib import Path
from credit_risk.dataset import AFTER_EDA, load_splits
from credit_risk.features import prep_one_split, load_features, FEATURES_DIR, DROP_COLS
from credit_risk.evaluation import compute_metrics
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    average_precision_score,
    brier_score_loss
)

In [2]:
train_df, val_df, test_df, _ = load_splits(path=AFTER_EDA)

2026-06-20 15:22:11.033 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-06-20 15:22:11.044 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-06-20 15:22:11.550 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [3]:
feature_splits = load_features(path=FEATURES_DIR)

2026-06-20 15:22:12.091 | INFO     | credit_risk.features:load_features:263 - Loading the processed features...
2026-06-20 15:22:12.444 | INFO     | credit_risk.features:load_features:270 - Loaded successfully!


In [4]:
X_test = feature_splits['test'][0].to_numpy()

In [5]:
cwd = Path.cwd()
project_root = cwd.parent
model_path = project_root / 'models' / 'tuned_xgb'

In [6]:
tuned_xgb_model = joblib.load(model_path / 'model.pkl')

In [7]:
with open(model_path / 'metric.json', 'r') as f:
    metrics = json.load(f)

In [8]:
metrics

{'threshold': 0.16,
 'train': {'ROC-AUC': 0.7439300157154369,
  'PR-AUC': 0.381689819454631,
  'brier_score': 0.12316741794347763,
  'precision': 0.2783728747025197,
  'recall': 0.7311166956633803,
  'confusion_matrix': [[241383, 147064], [20864, 56731]]},
 'val': {'ROC-AUC': 0.708335490425068,
  'PR-AUC': 0.34885062650079623,
  'brier_score': 0.13714288175106049,
  'precision': 0.2789071369106183,
  'recall': 0.7163796119902025,
  'confusion_matrix': [[200124, 142917], [21885, 55278]]},
 'test': {'ROC-AUC': 0.6935841329758616,
  'PR-AUC': 0.3104578352376564,
  'brier_score': 0.13049693405628204,
  'precision': 0.2591251104930686,
  'recall': 0.6604027583175361,
  'confusion_matrix': [[221458, 137456], [24722, 48076]]}}

In [9]:
test_proba = tuned_xgb_model[1].predict_proba(X_test)[:,1]

In [10]:
y_pred = (test_proba >= metrics['threshold']).astype(int)

In [11]:
filtered_test_df, f_y_test = prep_one_split(df=test_df)

2026-06-20 15:23:34.561 | INFO     | credit_risk.features:prep_one_split:217 - Inside Function: prep_one_split
2026-06-20 15:23:34.561 | INFO     | credit_risk.features:sorting_with_issue_d:143 - Sorting the dataframe wrt to issue_d...
2026-06-20 15:23:34.829 | INFO     | credit_risk.features:sorting_with_issue_d:148 - Sorted successfully!
2026-06-20 15:23:34.830 | INFO     | credit_risk.features:split_target_and_features:153 - Inside Function: split_target_and_features
2026-06-20 15:23:34.830 | INFO     | credit_risk.features:split_target_and_features:154 - Splitting the target and the features...
2026-06-20 15:23:34.832 | INFO     | credit_risk.features:split_target_and_features:157 - features and target are splitted successfully...
2026-06-20 15:23:34.832 | INFO     | credit_risk.features:add_credit_yrs:170 - Inside Function: add_credit_yrs
2026-06-20 15:23:34.832 | INFO     | credit_risk.features:add_credit_yrs:172 - Adding the feature 'credit_yrs'
2026-06-20 15:23:34.836 | INFO   

In [12]:
y_pred.shape, f_y_test.shape

((431712,), (431712,))

In [13]:
y_pred = pd.Series(y_pred, index=f_y_test.index)

In [23]:
y_proba = pd.Series(test_proba, index=f_y_test.index)

In [16]:
# term
filtered_test_df['term'].value_counts()

term
36 months    321815
60 months    109897
Name: count, dtype: int64

In [19]:
filtered_test_df['target'] = f_y_test

In [20]:
# let's check if the segments of term has enough defaults
filtered_test_df.groupby(by='term')['target'].sum()

term
36 months    48951
60 months    23847
Name: target, dtype: int64

In [25]:
term = filtered_test_df['term']

In [28]:
term[term == ' 36 months'].index

Index([1095381, 1095383, 1095384, 1095387, 1095388, 1095389, 1083900, 1095391,
       1095392, 1095393,
       ...
       2181267, 2181266, 2181265, 2181264, 2181262, 2181252, 2181258, 2181254,
       2181260, 2176662],
      dtype='int64', length=321815)

In [43]:
def get_segment_metrics(feature: pd.Series, y_true: pd.Series, y_proba: pd.Series, threshold: float) -> list[dict[str, Any]]:
    """Gives a segment wise model performance for a given feature"""
    y_pred = pd.Series((y_proba >= threshold).astype(int), index=y_true.index)
    result = []
    segments = feature.value_counts().index.to_list()
    for segment in segments:
        metrics = {}
        seg_idx = feature[feature == segment].index

        seg_y_true = y_true.loc[seg_idx]
        seg_y_pred = y_pred.loc[seg_idx]
        seg_y_proba = y_proba.loc[seg_idx]
        if seg_y_true.sum() == 0 or seg_y_true.sum() == len(seg_y_true):
            continue
        seg_base_rate = (seg_y_true.sum()/len(seg_y_true))
        metrics['n'] = len(seg_idx)
        metrics['segment'] = segment
        metrics['n_positives'] = int(seg_y_true.sum())
        metrics['base_rate'] = seg_base_rate.item()
        metrics['ROC-AUC'] = roc_auc_score(y_true=seg_y_true, y_score=seg_y_proba)
        metrics['PR-AUC'] = average_precision_score(y_true=seg_y_true, y_score=seg_y_proba)
        metrics['mean_pred_proba'] = seg_y_proba.mean().item()
        metrics['brier_score'] = brier_score_loss(y_true=seg_y_true, y_proba=seg_y_proba)
        metrics['precision'] = precision_score(y_true=seg_y_true, y_pred=seg_y_pred)
        metrics['recall'] = recall_score(y_true=seg_y_true, y_pred=seg_y_pred)
        
        result.append(metrics)
        
    return result

In [44]:
term_metrics = get_segment_metrics(feature=filtered_test_df['term'], y_true=f_y_test, y_proba=y_proba, threshold=metrics['threshold'])

In [72]:
term_metrics = pd.DataFrame(term_metrics)

In [73]:
term_metrics

,n,segment,n_positives,base_rate,ROC-AUC,PR-AUC,mean_pred_proba,brier_score,precision,recall
0,321815,36 months,48951,0.152109,0.693465,0.281974,0.135752,0.121244,0.260446,0.539785
1,109897,60 months,23847,0.216994,0.692726,0.373932,0.253454,0.157591,0.257532,0.907997


In [51]:
_, _, test_df_raw, _ = load_splits(path=AFTER_EDA)
filtered_test_df, _ = prep_one_split(test_df_raw, drop_cols=[c for c in DROP_COLS if c != 'application_type'])
app_type = filtered_test_df['application_type']

2026-06-20 16:48:20.605 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-06-20 16:48:20.610 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-06-20 16:48:21.026 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)
2026-06-20 16:48:21.028 | INFO     | credit_risk.features:prep_one_split:217 - Inside Function: prep_one_split
2026-06-20 16:48:21.028 | INFO     | credit_risk.features:sorting_with_issue_d:143 - Sorting the dataframe wrt to issue_d...
2026-06-20 16:48:21.296 | INFO     | credit_risk.features:sorting_with_issue_d:148 - Sorted successfully!
2026-06-20 16:48:21.296 | INFO     | credit_risk.features:split_target_and_features:153 - Inside Function: split_target_and_features
2026-06-20 16:48:21.296 | INFO     | credit_risk.features:split_target_and_features:154 - Splitting

In [53]:
app_type.index.equals(y_proba.index)

True

In [54]:
app_type_metrics = get_segment_metrics(feature=app_type, y_true=f_y_test, y_proba=y_proba, threshold=metrics['threshold'])

In [74]:
app_type_metrics = pd.DataFrame(app_type_metrics)
app_type_metrics

,n,segment,n_positives,base_rate,ROC-AUC,PR-AUC,mean_pred_proba,brier_score,precision,recall
0,423032,Individual,71381,0.168737,0.694602,0.313025,0.164212,0.130392,0.260779,0.656267
1,8680,Joint App,1417,0.163249,0.686665,0.285320,0.238919,0.135597,0.208750,0.868737


In [57]:
home_ownership_metrics = get_segment_metrics(feature=filtered_test_df['home_ownership'], y_true=f_y_test, y_proba=y_proba, threshold=metrics['threshold'])

In [75]:
home_ownership_metrics = pd.DataFrame(data=home_ownership_metrics)
home_ownership_metrics

,n,segment,n_positives,base_rate,ROC-AUC,PR-AUC,mean_pred_proba,brier_score,precision,recall
0,210252,MORTGAGE,30118,0.143247,0.694521,0.273050,0.149285,0.115150,0.236304,0.598081
1,168655,RENT,33738,0.200042,0.684574,0.345342,0.186792,0.148947,0.280672,0.721205
2,52695,OWN,8924,0.169352,0.677876,0.294152,0.163890,0.132685,0.253310,0.640968
3,110,ANY,18,0.163636,0.737923,0.315382,0.126839,0.128258,0.354839,0.611111


In [64]:
# annual_inc
annual_inc_bins = pd.cut(
    filtered_test_df['annual_inc'],
    bins=[0, 40_000, 80_000, 120_000, float('inf')],
    labels=['<$40K', '$40K-$80K', '$80K-$120K', '>$120K']
)

In [66]:
annual_inc_metrics = get_segment_metrics(feature=annual_inc_bins, y_true=f_y_test, y_proba=y_proba, threshold=metrics['threshold'])

In [68]:
annual_inc_metrics = pd.DataFrame(annual_inc_metrics)
annual_inc_metrics

,n,segment,n_positives,base_rate,ROC-AUC,PR-AUC,mean_pred_proba,brier_score,precision,recall
0,208784,$40K-$80K,36909,0.176781,0.691732,0.320984,0.173468,0.135345,0.265357,0.678723
1,96273,$80K-$120K,14341,0.148962,0.699831,0.292375,0.144506,0.117971,0.254371,0.578272
2,72497,<$40K,14754,0.203512,0.659673,0.324034,0.199890,0.153548,0.260759,0.769622
3,54101,>$120K,6783,0.125377,0.699006,0.249933,0.127568,0.103144,0.225513,0.496241


In [69]:
filtered_test_df['addr_state']

1096141    TX
1095380    AK
1095381    TX
1095382    FL
1095383    NJ
           ..
2181256    CA
2181255    NC
2181254    CA
2181260    MS
2176662    AL
Name: addr_state, Length: 431712, dtype: str

In [71]:
purpose_metrics = get_segment_metrics(feature=filtered_test_df['purpose'], y_true=f_y_test, y_proba=y_proba, threshold=metrics['threshold'])
purpose_metrics = pd.DataFrame(purpose_metrics)
purpose_metrics

,n,segment,n_positives,base_rate,ROC-AUC,PR-AUC,mean_pred_proba,brier_score,precision,recall
0,247233,debt_consolidation,44413,0.179640,0.689074,0.322503,0.177747,0.137183,0.262662,0.700516
1,91101,credit_card,13263,0.145586,0.701662,0.281312,0.141701,0.116076,0.249725,0.564804
2,30996,home_improvement,4482,0.144599,0.694132,0.271452,0.142818,0.116273,0.242974,0.565149
3,28311,other,4687,0.165554,0.679527,0.292718,0.164814,0.130043,0.249034,0.646042
4,10354,major_purchase,1611,0.155592,0.702368,0.304601,0.145473,0.121917,0.263500,0.578523
5,5417,medical,969,0.178881,0.660723,0.291051,0.166415,0.140033,0.258441,0.639835
6,4788,car,601,0.125522,0.686866,0.227334,0.132964,0.104823,0.216538,0.514143
7,4754,small_business,1138,0.239377,0.674585,0.393404,0.234007,0.168889,0.289543,0.873462
8,3249,vacation,530,0.163127,0.693098,0.302024,0.145527,0.127937,0.271416,0.567925
9,3213,moving,627,0.195145,0.685527,0.348596,0.175955,0.146071,0.279683,0.676236


In [76]:
emp_length_metrics = get_segment_metrics(feature=filtered_test_df['emp_length'], y_true=f_y_test, y_proba=y_proba, threshold=metrics['threshold'])
emp_length_metrics = pd.DataFrame(emp_length_metrics)
emp_length_metrics

,n,segment,n_positives,base_rate,ROC-AUC,PR-AUC,mean_pred_proba,brier_score,precision,recall
0,149054,10+ years,23016,0.154414,0.695538,0.296414,0.157487,0.121687,0.248273,0.627781
1,39372,2 years,6757,0.171619,0.691401,0.311288,0.165022,0.132498,0.259672,0.651621
2,34533,3 years,5969,0.172849,0.685155,0.305834,0.165845,0.133984,0.260097,0.652706
3,31728,< 1 year,5562,0.175303,0.687767,0.309753,0.167197,0.135253,0.265215,0.661273
4,28972,1 year,5197,0.179380,0.685607,0.317168,0.169054,0.137604,0.266317,0.664229
5,26297,5 years,4483,0.170476,0.702547,0.321256,0.167093,0.130700,0.266009,0.679233
6,25659,4 years,4293,0.167310,0.697774,0.317948,0.167165,0.129031,0.258033,0.671558
7,18637,8 years,3099,0.166282,0.699572,0.311128,0.165128,0.128662,0.260347,0.663763
8,18522,6 years,2996,0.161754,0.703234,0.306993,0.165716,0.125737,0.252426,0.668558
9,16794,9 years,2750,0.163749,0.694429,0.309096,0.165209,0.127457,0.252116,0.649818
